[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Transactions &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations and their year of readings, and opens
two connections that the tasks share: `conn`, which makes changes, and `other`, which stands for
every other program reading the file. Run it first. The tasks do not depend on one another, and the
last cell closes both connections and removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
INSERT_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


conn = sqlite3.connect(DATABASE)
conn.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
station_ids = {name: conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
               for name, latitude in LATITUDES.items()}
conn.executemany(INSERT_READING, ((station_ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
conn.commit()
other = sqlite3.connect(DATABASE)


def count_readings(connection, station, day):
    """How many readings a station has on a day, as this connection sees the database."""
    return connection.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND hour LIKE ?",
                              (station_ids[station], f"{day}%")).fetchone()[0]


print("built", DATABASE)


built scratch/stations.db


**1.** Before and after a commit.


In [2]:
conn.execute(INSERT_READING, (station_ids["Oslo"], "2026-01-01T00:00", -3.0))
print("before commit:", conn.in_transaction, "and the other connection counts", count_readings(other, "Oslo", "2026-01-01"))

conn.commit()
print("after commit: ", conn.in_transaction, "and the other connection counts", count_readings(other, "Oslo", "2026-01-01"))


before commit: True and the other connection counts 0
after commit:  False and the other connection counts 1


The insert opened a transaction, and until the commit the row existed only for `conn`. The commit
ended the transaction and made the row visible to `other`.


**2.** A delete, rolled back.


In [3]:
conn.execute("DELETE FROM readings WHERE station_id = ? AND hour LIKE ?", (station_ids["Tromso"], "2025-01-01%"))
print("after the delete:  ", count_readings(conn, "Tromso", "2025-01-01"))

conn.rollback()
print("after the rollback:", count_readings(conn, "Tromso", "2025-01-01"))


after the delete:   0
after the rollback: 24


Even `conn` saw the day empty after the delete, but the delete was never committed, so `rollback`
restored all 24 readings.


**3.** A `with` block that raises.


In [4]:
try:
    with conn:
        conn.execute(INSERT_READING, (station_ids["Bergen"], "2026-01-02T00:00", 3.9))
        raise RuntimeError("the second reading never arrived")
except RuntimeError as error:
    print("caught:", error)

print("Bergen readings on 2 January:", count_readings(other, "Bergen", "2026-01-02"))
print("in a transaction:", conn.in_transaction)


caught: the second reading never arrived
Bergen readings on 2 January: 0
in a transaction: False


The exception left the `with` block, which rolled back the first insert before the `except` caught
it, so nothing from the block was saved and no transaction was left open.


**4.** A savepoint inside `BEGIN`.


In [5]:
conn.execute("BEGIN")
conn.execute(INSERT_READING, (station_ids["Bergen"], "2026-01-03T00:00", 3.8))
conn.execute(INSERT_READING, (station_ids["Bergen"], "2026-01-03T01:00", 3.6))
conn.execute("SAVEPOINT third")
conn.execute(INSERT_READING, (station_ids["Bergen"], "2026-01-03T02:00", 3.5))
conn.execute("ROLLBACK TO third")
conn.execute("RELEASE third")
conn.commit()

print("Bergen readings on 3 January:", count_readings(other, "Bergen", "2026-01-03"))


Bergen readings on 3 January: 2


`ROLLBACK TO third` undid only the insert after the savepoint, and the transaction stayed open for
the commit that saved the other two. `BEGIN` came first, so releasing the savepoint did not commit
on its own.


**5.** The commit inside `executescript`.


In [6]:
conn.execute(INSERT_READING, (station_ids["Oslo"], "2026-01-04T00:00", -3.3))
print("in a transaction before executescript:", conn.in_transaction)

conn.executescript("CREATE TABLE IF NOT EXISTS audit (note TEXT NOT NULL);")
print("in a transaction after executescript: ", conn.in_transaction)

conn.rollback()
print("the other connection counts:", count_readings(other, "Oslo", "2026-01-04"))


in a transaction before executescript: True
in a transaction after executescript:  False
the other connection counts: 1


`executescript` committed the pending insert before it created the table, so the rollback had nothing
to undo and the other connection counted the row.


**6.** A replacement that happens completely or not at all.


In [7]:
def replace_reading(connection, station, hour, celsius):
    """Replace a station's reading for an hour in one transaction, refusing an implausible value."""
    with connection:
        connection.execute("DELETE FROM readings WHERE station_id = ? AND hour = ?", (station_ids[station], hour))
        if not -60 <= celsius <= 60:
            raise ValueError(f"{celsius} is not a plausible temperature")
        connection.execute(INSERT_READING, (station_ids[station], hour, celsius))


reading = "SELECT celsius FROM readings WHERE station_id = ? AND hour = ?"
print("before:", other.execute(reading, (station_ids["Oslo"], "2025-01-01T00:00")).fetchall())
try:
    replace_reading(conn, "Oslo", "2025-01-01T00:00", 99.0)
except ValueError as error:
    print("refused:", error)
print("after the refused replacement:", other.execute(reading, (station_ids["Oslo"], "2025-01-01T00:00")).fetchall())

replace_reading(conn, "Oslo", "2025-01-01T00:00", -3.4)
print("after a replacement that finished:", other.execute(reading, (station_ids["Oslo"], "2025-01-01T00:00")).fetchall())


before: [(-3.5,)]
refused: 99.0 is not a plausible temperature
after the refused replacement: [(-3.5,)]
after a replacement that finished: [(-3.4,)]


The delete had already run when `ValueError` was raised, and the `with` block rolled it back, so the
old reading was still there. The replacement that finished committed the delete and the insert
together.

Last, close both connections and remove the scratch folder:


In [8]:
other.close()
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/08-transactions.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
